In [3]:
import json
import os
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, SKOS, DCTERMS

# -------------------------------
# Configuration
# -------------------------------
CACHE_FILE = "dcat_us_3_model.json"
USE_CACHE = True      # Set to False to force reload of online files

SHACL_URL = "https://raw.githubusercontent.com/DOI-DO/dcat-us/main/shacl/dcat-us_3.0_shacl_shapes.ttl"

VOCAB_URLS = [
    "https://www.dublincore.org/specifications/dublin-core/dcmi-terms/dublin_core_terms.ttl",
    "https://www.w3.org/ns/dcat.ttl",
    "http://xmlns.com/foaf/spec/index.rdf",
    "https://www.w3.org/2009/08/skos-reference/skos.rdf",
    "http://www.w3.org/2000/01/rdf-schema"
]

SH = Namespace("http://www.w3.org/ns/shacl#")


# -------------------------------
# Load from cache (FAST)
# -------------------------------
def load_cached_model():
    if os.path.exists(CACHE_FILE):
        print(f"Loading cached model from {CACHE_FILE} ...")
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return None


# -------------------------------
# Save model to cache
# -------------------------------
def save_cache(model_dict):
    print(f"Saving model to {CACHE_FILE} ...")
    with open(CACHE_FILE, "w") as f:
        json.dump(model_dict, f, indent=2)


# -------------------------------
# Build the model dictionary (SLOW)
# -------------------------------
def build_model_dict():

    # Load SHACL
    print("Loading DCAT-US 3 SHACL...")
    shacl_graph = Graph()
    shacl_graph.parse(SHACL_URL, format="turtle")
    print(f"Loaded {len(shacl_graph)} SHACL triples.\n")

    # Load vocabularies
    print("Loading vocabularies...")
    vocab_graph = Graph()
    for url in VOCAB_URLS:
        try:
            vocab_graph.parse(url)
            print(f"Loaded: {url}")
        except Exception as e:
            print(f"FAILED to load {url}: {e}")

    print(f"\nTotal vocabulary triples: {len(vocab_graph)}\n")

    # Extract NodeShapes
    node_shapes = list(shacl_graph.subjects(RDF.type, SH.NodeShape))
    print(f"Found {len(node_shapes)} NodeShapes.\n")

    model = {}

    # For each NodeShape, extract properties + definitions
    for shape in node_shapes:

        shape_dict = {"properties": []}

        for prop_block in shacl_graph.objects(shape, SH.property):

            path = shacl_graph.value(prop_block, SH.path)
            if not path:
                continue

            # SHACL constraints
            min_count = shacl_graph.value(prop_block, SH.minCount)
            max_count = shacl_graph.value(prop_block, SH.maxCount)
            datatype = shacl_graph.value(prop_block, SH.datatype)
            klass = shacl_graph.value(prop_block, SH["class"])

            # Vocabulary definitions
            label = vocab_graph.value(path, RDFS.label)
            comment = vocab_graph.value(path, RDFS.comment)
            definition = vocab_graph.value(path, SKOS.definition)
            description = vocab_graph.value(path, DCTERMS.description)

            # Build property entry
            prop_entry = {
                "path": str(path),
                "minCount": int(min_count) if min_count else None,
                "maxCount": int(max_count) if max_count else None,
                "datatype": str(datatype) if datatype else None,
                "class": str(klass) if klass else None,
                "label": str(label) if label else None,
                "comment": str(comment) if comment else None,
                "definition": str(definition) if definition else None,
                "description": str(description) if description else None
            }

            shape_dict["properties"].append(prop_entry)

        model[str(shape)] = shape_dict

    return model


# -------------------------------
# Main loader
# -------------------------------
def load_dcat_us_3_model(force_rebuild=False):

    if USE_CACHE and not force_rebuild:
        cached = load_cached_model()
        if cached:
            return cached

    # Build slow version
    model = build_model_dict()
    save_cache(model)

    return model


# -------------------------------
# Run
# -------------------------------
if __name__ == "__main__":
    model = load_dcat_us_3_model()
    print("\nModel loaded. NodeShapes:")
    for shape in list(model.keys())[:10]:
        print(" -", shape)
    print("\nTotal NodeShapes:", len(model))


Loading DCAT-US 3 SHACL...
Loaded 4042 SHACL triples.

Loading vocabularies...
Loaded: https://www.dublincore.org/specifications/dublin-core/dcmi-terms/dublin_core_terms.ttl
Loaded: https://www.w3.org/ns/dcat.ttl
FAILED to load http://xmlns.com/foaf/spec/index.rdf: <urlopen error [Errno 110] Connection timed out>
Loaded: https://www.w3.org/2009/08/skos-reference/skos.rdf
Loaded: http://www.w3.org/2000/01/rdf-schema

Total vocabulary triples: 2734

Found 34 NodeShapes.

Saving model to dcat_us_3_model.json ...

Model loaded. NodeShapes:
 - http://data.resources.gov/shapes/dcat-us#AccessRestriction_Shape
 - http://data.resources.gov/shapes/dcat-us#Activity_Shape
 - http://data.resources.gov/shapes/dcat-us#Address_Contact_Shape
 - http://data.resources.gov/shapes/dcat-us#Address_Location_Shape
 - http://data.resources.gov/shapes/dcat-us#Agent_Shape
 - http://data.resources.gov/shapes/dcat-us#Attribution_Shape
 - http://data.resources.gov/shapes/dcat-us#CUIRestriction
 - http://data.resour

In [6]:
for key in model.keys():
    print(key)

http://data.resources.gov/shapes/dcat-us#AccessRestriction_Shape
http://data.resources.gov/shapes/dcat-us#Activity_Shape
http://data.resources.gov/shapes/dcat-us#Address_Contact_Shape
http://data.resources.gov/shapes/dcat-us#Address_Location_Shape
http://data.resources.gov/shapes/dcat-us#Agent_Shape
http://data.resources.gov/shapes/dcat-us#Attribution_Shape
http://data.resources.gov/shapes/dcat-us#CUIRestriction
http://data.resources.gov/shapes/dcat-us#CatalogRecord_Shape
http://data.resources.gov/shapes/dcat-us#Catalog_Shape
http://data.resources.gov/shapes/dcat-us#Checksum_Shape
http://data.resources.gov/shapes/dcat-us#ConceptScheme_Shape
http://data.resources.gov/shapes/dcat-us#Concept_Shape
http://data.resources.gov/shapes/dcat-us#DataService_Shape
http://data.resources.gov/shapes/dcat-us#DatasetSeries_Shape
http://data.resources.gov/shapes/dcat-us#Dataset_Shape
http://data.resources.gov/shapes/dcat-us#Distribution_Shape
http://data.resources.gov/shapes/dcat-us#Document_Shape
http: